# Params Tuning

In [1]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras

from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from sklearn.model_selection import GridSearchCV
from tensorflow.keras import backend as k

from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt


## Data Loading

In [2]:
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 

X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

y = y["Diagnosis"].map({"B": 0, "M": 1}).to_frame()

## Neural Network Structure Creating

In [3]:
def create_network(optimizer, loss, kernel_initializer, activation, neurons):
    k.clear_session()
    neural_network = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2), # dropout 20%
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2), # dropout 20%
        tf.keras.layers.Dense(units=1, activation='sigmoid')    
    ])
    # maps the optimizer name received from the grid to the optimizer object
    optimizers = {
        'adam': tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5),
        'sgd': tf.keras.optimizers.SGD(learning_rate=0.01, clipvalue=0.5),
    }
    optimizer = optimizers[optimizer]
    neural_network.compile(optimizer=optimizer, loss=loss, metrics=['binary_accuracy'])
    return neural_network

In [4]:
neural_network = KerasClassifier(model=create_network, epochs=100, batch_size=10)

### Normalization

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(StandardScaler(), neural_network)

In [6]:
params = {
    "kerasclassifier__batch_size": [10, 30],
    "kerasclassifier__epochs": [50, 100],
    "kerasclassifier__model__optimizer": ['adam', 'sgd'],
    "kerasclassifier__model__loss": ['binary_crossentropy'], # hinge removed: it expects -1/+1 labels and no sigmoid
    "kerasclassifier__model__kernel_initializer": ['random_uniform', 'normal'],
    "kerasclassifier__model__activation": ['relu', 'tanh'],
    "kerasclassifier__model__neurons": [16, 8]
}
params

{'kerasclassifier__batch_size': [10, 30],
 'kerasclassifier__epochs': [50, 100],
 'kerasclassifier__model__optimizer': ['adam', 'sgd'],
 'kerasclassifier__model__loss': ['binary_crossentropy'],
 'kerasclassifier__model__kernel_initializer': ['random_uniform', 'normal'],
 'kerasclassifier__model__activation': ['relu', 'tanh'],
 'kerasclassifier__model__neurons': [16, 8]}

In [7]:
grid_search = GridSearchCV(estimator = pipeline, param_grid = params, scoring = 'accuracy', cv=5)

In [8]:
grid_search

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...141E52B60>))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kerasclassifier__batch_size': [10, 30], 'kerasclassifier__epochs': [50, 100], 'kerasclassifier__model__activation': ['relu', 'tanh'], 'kerasclassifier__model__kernel_initializer': ['random_uniform', 'normal'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcom

In [9]:
grid_search.fit(X, y.values.ravel())


Epoch 1/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.9121 - loss: 0.6139
Epoch 2/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9648 - loss: 0.3089
Epoch 3/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9692 - loss: 0.1372
Epoch 4/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9736 - loss: 0.0985
Epoch 5/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9736 - loss: 0.0854
Epoch 6/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9868 - loss: 0.0725
Epoch 7/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.9824 - loss: 0.0723
Epoch 8/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9780 - loss: 0.0630
Epoch 9/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9846 - loss: 0.0701
Epoch 10/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9846 - loss: 0.0491
Epoch 11/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.9890 - loss: 0.0

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...141E52B60>))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kerasclassifier__batch_size': [10, 30], 'kerasclassifier__epochs': [50, 100], 'kerasclassifier__model__activation': ['relu', 'tanh'], 'kerasclassifier__model__kernel_initializer': ['random_uniform', 'normal'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcom

In [10]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('kerasclassifier',
                                        KerasClassifier(batch_size=10, epochs=100, model=<function create_network at 0x000001D141E52B60>))]),
             param_grid={'kerasclassifier__batch_size': [10, 30],
                         'kerasclassifier__epochs': [50, 100],
                         'kerasclassifier__model__activation': ['relu', 'tanh'],
                         'kerasclassifier__model__kernel_initializer': ['random_uniform',
                                                                        'normal'],
                         'kerasclassifier__model__loss': ['binary_crossentropy'],
                         'kerasclassifier__model__neurons': [16, 8],
                         'kerasclassifier__model__optimizer': ['adam', 'sgd']},
             scoring='accuracy')


In [12]:
best_params = grid_search.best_params_
print(best_params)

{'kerasclassifier__batch_size': 10, 'kerasclassifier__epochs': 50, 'kerasclassifier__model__activation': 'relu', 'kerasclassifier__model__kernel_initializer': 'random_uniform', 'kerasclassifier__model__loss': 'binary_crossentropy', 'kerasclassifier__model__neurons': 8, 'kerasclassifier__model__optimizer': 'sgd'}
